# 1 - Pull live data

Pull current public data from NYC Open Data and NYSDOT, then store timestamped local snapshots under `data/cache/`.

The API remains authoritative. Set `REFRESH = True` to force new requests; leave it `False` to reuse the latest local snapshot for the same query parameters.

In [1]:
from pathlib import Path
import sys

# Make imports work whether Jupyter starts in the repo root or notebooks/.
ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

CACHE_DIR = ROOT / "data" / "cache"
OUTPUT_DIR = ROOT / "data" / "outputs"
CACHE_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

ANALYSIS_START = "2021-01-01"
ANALYSIS_END = "2025-12-31"
REFRESH = False

print("Repo root:", ROOT)
print("Analysis period:", ANALYSIS_START, "to", ANALYSIS_END)
print("Force API refresh:", REFRESH)


Repo root: /Users/uyen-phuongnguyen/schoolgithub/worst-streets-watchlist
Analysis period: 2021-01-01 to 2025-12-31
Force API refresh: False


In [2]:
from src.crashes import fetch_crashes
from src.lion import fetch_lion, prepare_lion_segments
from src.aadt import fetch_aadt
from src.traffic import fetch_automated_traffic_counts, aggregate_traffic_by_segment
from src.api import save_cache


## Crash events

The initial corridor analysis uses the aggregate injury/fatality fields already present in the crash-event table. Person- and vehicle-level pulls are available in `src.crashes` but are optional enrichment because they are much larger.

In [3]:
crashes = fetch_crashes(
    ANALYSIS_START,
    ANALYSIS_END,
    refresh=REFRESH,
    cache_dir=CACHE_DIR,
)

print(f"Crashes: {len(crashes):,}")
print("Geocoded:", crashes[["latitude", "longitude"]].notna().all(axis=1).mean())
crashes.head()


Using cached crashes snapshot: crashes_ebb86205b8a8
Crashes: 487,931
Geocoded: 0.9214643054038378


,collision_id,crash_date,crash_time,borough,zip_code,latitude,longitude,on_street_name,cross_street_name,number_of_persons_injured,number_of_persons_killed,number_of_pedestrians_injured,number_of_pedestrians_killed,number_of_cyclist_injured,number_of_cyclist_killed,number_of_motorist_injured,number_of_motorist_killed,contributing_factor_vehicle_1,off_street_name
0,01687a2713154e8ddcb6f3549ef3ecc9,2021-01-01,06:45,MANHATTAN,10017,40.752377,-73.97035,EAST 46 STREET,2 AVENUE,0.0,0,0,0,0,0,0,0,Driver Inexperience,NaN
1,037f52da1afb7f5b35c8df50832990cb,2021-01-01,22:15,NaN,NaN,40.663227,-73.93159,UTICA AVENUE,EAST NEW YORK AVENUE,0.0,0,0,0,0,0,0,0,Unspecified,NaN
2,03bb1ab07c11636a66a22c8b8f4e4ec9,2021-01-01,00:00,BROOKLYN,11212,40.661873,-73.92878,NaN,NaN,0.0,0,0,0,0,0,0,0,Unspecified,108 EAST 91 STREET
3,041ce856d0ef932f1db43e343a71f8b2,2021-01-01,21:30,NaN,NaN,40.740467,-73.97295,FDR DRIVE,NaN,0.0,0,0,0,0,0,0,0,Passing or Lane Usage Improper,NaN
4,04ac53e7f3d94b9965d5de73a5bb3bff,2021-01-01,00:13,BROOKLYN,11208,40.660450,-73.86719,FLATLANDS AVENUE,FOUNTAIN AVENUE,2.0,0,0,0,0,0,2,0,Driver Inexperience,NaN


## LION street network

`prepare_lion_segments()` dissolves duplicate LION records so the analytical table has exactly one row per physical `SegmentID`.

In [ ]:
lion_raw = fetch_lion(refresh=REFRESH, cache_dir=CACHE_DIR)
lion_segments = prepare_lion_segments(lion_raw)

print(f"Raw LION rows: {len(lion_raw):,}")
print(f"Unique prepared segments: {len(lion_segments):,}")
lion_segments.head()


Using cached LION snapshot: lion_79875d901162


## NYSDOT AADT

This is the main motor-vehicle exposure source. The query is restricted to a broad NYC bounding box and cached locally.

In [5]:
aadt = fetch_aadt(refresh=REFRESH, cache_dir=CACHE_DIR)

print(f"AADT line features in NYC bounding box: {len(aadt):,}")
print(aadt[[c for c in ["roadway_name", "aadt", "calculation_year", "year_last_act"] if c in aadt.columns]].head())


HTTPError: 500 Server Error: Internal Server Error for url: https://gis.dot.ny.gov/hostingny/rest/services/Roadways/Traffic_Monitoring/FeatureServer/1/query?where=CountyFIPS+IN+%28%27005%27%2C+%27047%27%2C+%27061%27%2C+%27081%27%2C+%27085%27%29+AND+AADT+IS+NOT+NULL+AND+AADT+%3E+0&outFields=%2A&returnGeometry=true&outSR=4326&resultOffset=0&resultRecordCount=2000&orderByFields=OBJECTID&f=geojson

## NYC DOT automated traffic samples

These are 15-minute sampled counts, **not AADT**. They are useful as a supplemental/validation source. limits  pull to 2024–2025 to avoid downloading the full ~1.9M-row history.

In [ ]:
PULL_TRAFFIC_SAMPLES = False

if PULL_TRAFFIC_SAMPLES:
    traffic = fetch_automated_traffic_counts(
        years=[2024, 2025],
        refresh=REFRESH,
        cache_dir=CACHE_DIR,
    )
    traffic_summary = aggregate_traffic_by_segment(traffic)
    save_cache(traffic_summary, "traffic_summary_2024_2025", cache_dir=CACHE_DIR)
    print(f"Traffic rows: {len(traffic):,}")
    print(f"Segments with sampled traffic: {len(traffic_summary):,}")
    display(traffic_summary.head())
else:
    print("Skipped. Set PULL_TRAFFIC_SAMPLES = True to pull sampled ATR counts.")


## DO NOT RUN YET NEED TO DEBUG person / vehicle enrichment

 age, pedestrian-action, vehicle-type, or other record-level attributes. **Aggregate each table to one row per collision before joining to crashes** to avoid many-to-many duplication.

```python
from src.crashes import (
    fetch_people, aggregate_people_by_crash,
    fetch_vehicles, aggregate_vehicles_by_crash,
)

people = fetch_people(ANALYSIS_START, ANALYSIS_END, cache_dir=CACHE_DIR)
person_by_crash = aggregate_people_by_crash(people)

vehicles = fetch_vehicles(ANALYSIS_START, ANALYSIS_END, cache_dir=CACHE_DIR)
vehicle_by_crash = aggregate_vehicles_by_crash(vehicles)
```
